<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/notebook4_Master_Patient_Index_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Master Patient Index Generator (PPMI)

**Objective:** merge `participant_index.csv`, `imagenes_index.csv`, and `exomas_index.csv` into a single `paciente_master_index.csv`, one row per PATNO. It does not open any DICOMs, any VCFs, it does not traverse RARs or TARs — it works exclusively with pandas on the already generated CSVs. It should run in seconds.

**Compatibility Note:** this notebook is defensive regarding column names (it uses several possible aliases) because the actual indices generated by notebooks 1-3 do not always exactly match the ideal schema described in the project design. Where data does not exist in any source, it is explicitly left as `"not available"` instead of being invented.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Path Configuration

In [ ]:
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"
RESULTADOS_DIR = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados"

RUTA_PARTICIPANT_INDEX = RESULTADOS_DIR + "/participant_index.csv"
RUTA_IMAGENES_INDEX = RESULTADOS_DIR + "/imagenes_index.csv"
RUTA_EXOMAS_INDEX = RESULTADOS_DIR + "/exomas_index.csv"

# Optional: cache of individual paths within the RAR (generated the first time
# by the extraction/visualization notebook). If it exists, it is used to enrich the
# master index with specific DICOM paths; if it does not exist, those fields remain
# marked as "not available" without blocking the generation of the master index.
RUTA_CACHE_RUTAS_DICOM = RESULTADOS_DIR + "/rutas_dicom_completas.csv"

RUTA_SALIDA_MASTER_INDEX = RESULTADOS_DIR + "/paciente_master_index.csv"

for r in [RUTA_IMAGENES_INDEX, RUTA_EXOMAS_INDEX]:
    assert os.path.exists(r), f"Not found: {r}"
print("Base indexes verified OK.")

Índices base verificados OK.


## 1. Loading the three indexes

`participant_index.csv` is treated as optional: if it does not yet exist, the master index is still built from images + exomes, and purely clinical fields remain as `"not available"`.

In [ ]:
df_participant = pd.read_csv(RUTA_PARTICIPANT_INDEX) if os.path.exists(RUTA_PARTICIPANT_INDEX) else None
df_imagenes = pd.read_csv(RUTA_IMAGENES_INDEX)
df_exomas = pd.read_csv(RUTA_EXOMAS_INDEX)
df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM) if os.path.exists(RUTA_CACHE_RUTAS_DICOM) else None

print("participant_index.csv:", "loaded (%d rows)" % len(df_participant) if df_participant is not None else "NOT found")
print("imagenes_index.csv:   loaded (%d rows)" % len(df_imagenes))
print("exomas_index.csv:     loaded (%d rows)" % len(df_exomas))
print("rutas_dicom_completas.csv (optional):", "loaded (%d rows)" % len(df_rutas_dicom) if df_rutas_dicom is not None else "not found (specific DICOM paths are omitted)")

/tmp/ipykernel_15751/3179111131.py:4: DtypeWarning: Columns (3,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM) if os.path.exists(RUTA_CACHE_RUTAS_DICOM) else None


participant_index.csv: cargado (8619 filas)
imagenes_index.csv:   cargado (6441 filas)
exomas_index.csv:     cargado (645 filas)
rutas_dicom_completas.csv (opcional): cargado (3277408 filas)


## 2. Type Normalization

`PATNO` must remain an integer in all four sources to allow cross-referencing without type errors (float vs int vs str).

In [ ]:
def normalizar_patno(df, columna="PATNO"):
    if df is None or columna not in df.columns:
        return df
    df = df.copy()
    df[columna] = pd.to_numeric(df[columna], errors="coerce")
    df = df.dropna(subset=[columna])
    df[columna] = df[columna].astype(int)
    return df

df_participant = normalizar_patno(df_participant)
df_imagenes = normalizar_patno(df_imagenes)
df_exomas = normalizar_patno(df_exomas)
df_rutas_dicom = normalizar_patno(df_rutas_dicom)

print("Types normalized.")

Tipos normalizados.


## 3. Aggregation of images per patient

`imagenes_index.csv` has one row per study, not per patient. It is aggregated at the PATNO level.

**Column mapping used** (adjust here if your actual schema differs):
- Found orientations → `vista` column (e.g., AXIAL / SAGITAL)
- Found modalities → `secuencia` column (e.g., T1 / T2)
- Found sequences/dimensions → `dimension` column (e.g., 2D / 3D)

In [ ]:
def _valores_unicos_texto(serie):
    valores = sorted(set(str(v) for v in serie.dropna().unique()))
    return ", ".join(valores) if valores else "not available"


def construir_agregado_imagenes(df_img):
    if df_img is None or df_img.empty:
        return pd.DataFrame(columns=[
            "PATNO", "Has_images", "N_total_dicom_files", "N_total_studies",
            "N_total_series", "Found_modalities", "Found_orientations",
            "Found_sequences", "First_study_path", "Image_diagnostic_group",
        ])

    col_n_archivos = "n_archivos" if "n_archivos" in df_img.columns else None
    col_estudio = "estudio" if "estudio" in df_img.columns else None
    col_vista = "vista" if "vista" in df_img.columns else None
    col_secuencia = "secuencia" if "secuencia" in df_img.columns else None
    col_dimension = "dimension" if "dimension" in df_img.columns else None
    col_grupo = "grupo_diagnostico" if "grupo_diagnostico" in df_img.columns else None

    filas = []
    for patno, grupo in df_img.groupby("PATNO"):
        filas.append({
            "PATNO": patno,
            "Has_images": True,
            "N_total_dicom_files": int(grupo[col_n_archivos].sum()) if col_n_archivos else "not available",
            "N_total_studies": len(grupo),
            "N_total_series": grupo[col_estudio].nunique() if col_estudio else "not available",
            "Found_modalities": _valores_unicos_texto(grupo[col_secuencia]) if col_secuencia else "not available",
            "Found_orientations": _valores_unicos_texto(grupo[col_vista]) if col_vista else "not available",
            "Found_sequences": _valores_unicos_texto(grupo[col_dimension]) if col_dimension else "not available",
            "First_study_path": grupo[col_estudio].dropna().iloc[0] if col_estudio and grupo[col_estudio].notna().any() else "not available",
            "Image_diagnostic_group": grupo[col_grupo].dropna().iloc[0] if col_grupo and grupo[col_grupo].notna().any() else "not available",
        })

    return pd.DataFrame(filas)


agregado_imagenes = construir_agregado_imagenes(df_imagenes)
print("Patients with aggregated images:", len(agregado_imagenes))

Pacientes con imágenes agregados: 1471


,PATNO,Tiene_imagenes,N_archivos_dicom_total,N_estudios_total,N_series_total,Modalidades_encontradas,Orientaciones_encontradas,Secuencias_encontradas,Ruta_primer_estudio,Grupo_diagnostico_imagenes
0,3000,True,170,2,2,"T1, T2","AXIAL, SAGITAL","2D, 3D",AX_T2_FLAIR,Control
1,3001,True,265,3,3,"T1, T2","AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,Parkinson_Disease
2,3002,True,207,3,3,"T1, T2","AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,Parkinson_Disease
3,3003,True,205,3,3,"T1, T2","AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,Parkinson_Disease
4,3004,True,257,3,3,"T1, T2","AXIAL, Axial, SAGITAL","2D, 3D",AX_T2_FLAIR_5_1,Control


## 4. Optional enrichment with specific DICOM paths

Only executed if `rutas_dicom_completas.csv` exists. Adds `Ruta_primer_dicom` and `Ruta_interna_rar` (full path within the RAR of the first file found per patient).

In [ ]:
def construir_rutas_puntuales(df_rutas):
    if df_rutas is None or df_rutas.empty or "ruta_completa" not in df_rutas.columns:
        return pd.DataFrame(columns=["PATNO", "First_dicom_path", "Internal_rar_path"])

    filas = []
    for patno, grupo in df_rutas.groupby("PATNO"):
        primera_ruta = grupo["ruta_completa"].iloc[0]
        filas.append({
            "PATNO": patno,
            "First_dicom_path": primera_ruta.split("/")[-1],
            "Internal_rar_path": primera_ruta,
        })
    return pd.DataFrame(filas)


rutas_puntuales = construir_rutas_puntuales(df_rutas_dicom)
if rutas_puntuales.empty:
    print("No individual path cache available: First_dicom_path and Internal_rar_path "
          "will remain 'not available' in the master index.")
else:
    print("Specific paths constructed for", len(rutas_puntuales), "patients.")

Rutas puntuales construidas para 1471 pacientes.


## 5. Exome aggregation per patient

`exomas_index.csv` already contains one row per patient; column names are normalized and duplicates are removed for safety.

In [ ]:
def construir_agregado_exomas(df_exo):
    if df_exo is None or df_exo.empty:
        return pd.DataFrame(columns=["PATNO", "Has_exome", "TAR_File", "Internal_VCF_Path", "VCF_File"])

    col_tar = next((c for c in ["archivo_tar", "TAR", "archivo_TAR"] if c in df_exo.columns), None)
    col_vcf = next((c for c in ["archivo_vcf", "VCF", "archivo_VCF"] if c in df_exo.columns), None)
    col_ruta_vcf = next((c for c in ["ruta_interna_vcf", "ruta_vcf", "ruta_interna"] if c in df_exo.columns), None)

    df_dedup = df_exo.drop_duplicates(subset=["PATNO"]).copy()

    df_dedup["Has_exome"] = True
    df_dedup["TAR_File"] = df_dedup[col_tar] if col_tar else "not available"
    df_dedup["VCF_File"] = df_dedup[col_vcf] if col_vcf else "not available"
    # If there is no explicit internal VCF path column, the VCF file name is used
    # as the best available approximation (a path with folders is not invented).
    if col_ruta_vcf:
        df_dedup["Internal_VCF_Path"] = df_dedup[col_ruta_vcf]
    elif col_vcf:
        df_dedup["Internal_VCF_Path"] = df_dedup[col_vcf]
    else:
        df_dedup["Internal_VCF_Path"] = "not available"

    return df_dedup[["PATNO", "Has_exome", "TAR_File", "Internal_VCF_Path", "VCF_File"]]


agregado_exomas = construir_agregado_exomas(df_exomas)
print("Patients with aggregated exomes:", len(agregado_exomas))

Pacientes con exoma agregados: 645


,PATNO,Tiene_exoma,Archivo_TAR,Ruta_interna_VCF,Archivo_VCF
0,3000,True,ppmi_wes_645_indiv_vcf_set_26_of_78.tar.gz,PPMI_SI_3000.raw.vcf,PPMI_SI_3000.raw.vcf
1,3001,True,ppmi_wes_645_indiv_vcf_set_22_of_78.tar.gz,PPMI_SI_3001.raw.vcf,PPMI_SI_3001.raw.vcf
2,3002,True,ppmi_wes_645_indiv_vcf_set_59_of_78.tar.gz,PPMI_SI_3002.raw.vcf,PPMI_SI_3002.raw.vcf
3,3003,True,ppmi_wes_645_indiv_vcf_set_18_of_78.tar.gz,PPMI_SI_3003.raw.vcf,PPMI_SI_3003.raw.vcf
4,3004,True,ppmi_wes_645_indiv_vcf_set_48_of_78.tar.gz,PPMI_SI_3004.raw.vcf,PPMI_SI_3004.raw.vcf


## 6. Clinical information per patient

Equally defensive with column names: it tries several reasonable aliases for each field.

In [ ]:
def construir_info_clinica(df_part):
    if df_part is None:
        return pd.DataFrame(columns=[
            "PATNO", "Alias_SI", "Enrollment_Status", "Sex", "Age", "Clinical_Group", "Diagnosis"
        ])

    mapeo = {
        "Alias_SI": ["Alias_SI", "Alias SI", "SI", "SITE_ID"],
        "Enrollment_Status": ["Estado_participante", "ENROLL_STATUS", "Estado"],
        "Sex": ["Sexo", "Sex", "SEX"],
        "Age": ["Edad", "Age"],
        "Clinical_Group": ["Grupo", "COHORT", "grupo_diagnostico"],
        "Diagnosis": ["Diagnostico", "Diagnosis", "DIAGNOSIS"],
    }

    resultado = pd.DataFrame({"PATNO": df_part["PATNO"]})
    for destino, candidatas in mapeo.items():
        col = next((c for c in candidatas if c in df_part.columns), None)
        resultado[destino] = df_part[col] if col else "not available"

    return resultado.drop_duplicates(subset=["PATNO"])


info_clinica = construir_info_clinica(df_participant)
print("Patients with clinical info:", len(info_clinica))

Pacientes con info clínica: 8619


,PATNO,Alias_SI,Estado_Enrolled,Sexo,Edad,Grupo_clinico,Diagnostico
0,3000,no disponible,Withdrew,no disponible,no disponible,2,no disponible
1,3001,no disponible,Withdrew,no disponible,no disponible,1,no disponible
2,3002,no disponible,Withdrew,no disponible,no disponible,1,no disponible
3,3003,no disponible,Enrolled,no disponible,no disponible,1,no disponible
4,3004,no disponible,Enrolled,no disponible,no disponible,2,no disponible


## 7. Master Index Construction

The universe of patients is the **union** of PATNOs present in any of the three sources (not just the intersection), so that a patient with exome but without images (or vice versa) is also represented.

In [ ]:
universo_patno = set()
for df in [info_clinica, agregado_imagenes, agregado_exomas]:
    if df is not None and "PATNO" in df.columns:
        universo_patno |= set(df["PATNO"])

paciente_master_index = pd.DataFrame({"PATNO": sorted(universo_patno)})

paciente_master_index = paciente_master_index.merge(info_clinica, on="PATNO", how="left")
paciente_master_index = paciente_master_index.merge(agregado_imagenes, on="PATNO", how="left")
paciente_master_index = paciente_master_index.merge(rutas_puntuales, on="PATNO", how="left")
paciente_master_index = paciente_master_index.merge(agregado_exomas, on="PATNO", how="left")

# Normalize boolean flags and fill empty values without inventing data
paciente_master_index["Has_images"] = paciente_master_index["Has_images"].fillna(False)
paciente_master_index["Has_exome"] = paciente_master_index["Has_exome"].fillna(False)
paciente_master_index["Has_both"] = paciente_master_index["Has_images"] & paciente_master_index["Has_exome"]

columnas_texto_default = [
    "Alias_SI", "Enrollment_Status", "Sex", "Age", "Clinical_Group", "Diagnosis",
    "Found_modalities", "Found_orientations", "Found_sequences",
    "First_study_path", "Image_diagnostic_group", "First_dicom_path",
    "Internal_rar_path", "TAR_File", "Internal_VCF_Path", "VCF_File",
]
for col in columnas_texto_default:
    if col in paciente_master_index.columns:
        paciente_master_index[col] = paciente_master_index[col].fillna("not available")

columnas_numericas_default = ["N_total_dicom_files", "N_total_studies", "N_total_series"]
for col in columnas_numericas_default:
    if col in paciente_master_index.columns:
        paciente_master_index[col] = paciente_master_index[col].fillna(0)

orden_columnas = [
    "PATNO", "Alias_SI", "Enrollment_Status", "Sex", "Age", "Clinical_Group", "Diagnosis",
    "Has_images", "N_total_dicom_files", "N_total_studies", "N_total_series",
    "Found_modalities", "Found_orientations", "Found_sequences",
    "First_study_path", "First_dicom_path", "Internal_rar_path",
    "Has_exome", "TAR_File", "Internal_VCF_Path", "VCF_File",
    "Has_both",
]
orden_columnas = [c for c in orden_columnas if c in paciente_master_index.columns]
paciente_master_index = paciente_master_index[orden_columnas]

print("Master index constructed:", paciente_master_index.shape)

Índice maestro construido: (8619, 22)


/tmp/ipykernel_15751/3282955901.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  paciente_master_index["Tiene_imagenes"] = paciente_master_index["Tiene_imagenes"].fillna(False)
/tmp/ipykernel_15751/3282955901.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  paciente_master_index["Tiene_exoma"] = paciente_master_index["Tiene_exoma"].fillna(False)


,PATNO,Alias_SI,Estado_Enrolled,Sexo,Edad,Grupo_clinico,Diagnostico,Tiene_imagenes,N_archivos_dicom_total,N_estudios_total,...,Orientaciones_encontradas,Secuencias_encontradas,Ruta_primer_estudio,Ruta_primer_dicom,Ruta_interna_rar,Tiene_exoma,Archivo_TAR,Ruta_interna_VCF,Archivo_VCF,Tiene_ambas
0,3000,no disponible,Withdrew,no disponible,no disponible,2,no disponible,True,170.0,2.0,...,"AXIAL, SAGITAL","2D, 3D",AX_T2_FLAIR,PPMI_3000_MR_sag_3D_FSPGR_BRAVO_straight__br_r...,Imagenes_PPMI/Control/T1/SAGITAL/3D/CONTROL_T1...,True,ppmi_wes_645_indiv_vcf_set_26_of_78.tar.gz,PPMI_SI_3000.raw.vcf,PPMI_SI_3000.raw.vcf,True
1,3001,no disponible,Withdrew,no disponible,no disponible,1,no disponible,True,265.0,3.0,...,"AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,PPMI_3001_MR_sag_3D_FSPGR_BRAVO_straight__br_r...,Imagenes_PPMI/Parkinson_Disease/T1/SAGITAL/3D/...,True,ppmi_wes_645_indiv_vcf_set_22_of_78.tar.gz,PPMI_SI_3001.raw.vcf,PPMI_SI_3001.raw.vcf,True
2,3002,no disponible,Withdrew,no disponible,no disponible,1,no disponible,True,207.0,3.0,...,"AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,PPMI_3002_MR_sag_3D_FSPGR_BRAVO_straight__br_r...,Imagenes_PPMI/Parkinson_Disease/T1/SAGITAL/3D/...,True,ppmi_wes_645_indiv_vcf_set_59_of_78.tar.gz,PPMI_SI_3002.raw.vcf,PPMI_SI_3002.raw.vcf,True
3,3003,no disponible,Enrolled,no disponible,no disponible,1,no disponible,True,205.0,3.0,...,"AXIAL, SAGITAL","2D, 3D",AX_T2_AC-PC_line_Entire_Brain,PPMI_3003_MR_sag_3D_FSPGR_BRAVO_straight__br_r...,Imagenes_PPMI/Parkinson_Disease/T1/SAGITAL/3D/...,True,ppmi_wes_645_indiv_vcf_set_18_of_78.tar.gz,PPMI_SI_3003.raw.vcf,PPMI_SI_3003.raw.vcf,True
4,3004,no disponible,Enrolled,no disponible,no disponible,2,no disponible,True,257.0,3.0,...,"AXIAL, Axial, SAGITAL","2D, 3D",AX_T2_FLAIR_5_1,PPMI_3004_MR_AX_T2_AC-PC_line_Entire_Brain__br...,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,True,ppmi_wes_645_indiv_vcf_set_48_of_78.tar.gz,PPMI_SI_3004.raw.vcf,PPMI_SI_3004.raw.vcf,True


## 8. Saving and final summary

In [ ]:
paciente_master_index.to_csv(RUTA_SALIDA_MASTER_INDEX, index=False)

total = len(paciente_master_index)
con_imagenes = int(paciente_master_index["Has_images"].sum())
con_exoma = int(paciente_master_index["Has_exome"].sum())
con_ambas = int(paciente_master_index["Has_both"].sum())
solo_imagenes = con_imagenes - con_ambas
solo_exoma = con_exoma - con_ambas
sin_ninguna = total - (con_imagenes + con_exoma - con_ambas)

print("=" * 50)
print("SUMMARY — paciente_master_index.csv")
print("=" * 50)
print(f"Total patients:            {total}")
print(f"Patients with images:       {con_imagenes}")
print(f"Patients with exome:          {con_exoma}")
print(f"Patients with both sources:  {con_ambas}")
print(f"Patients only images:      {solo_imagenes}")
print(f"Patients only exome:         {solo_exoma}")
print(f"Patients without any source: {sin_ninguna}")
print("=" * 50)
print("File generated:", RUTA_SALIDA_MASTER_INDEX)

RESUMEN — paciente_master_index.csv
Pacientes totales:            8619
Pacientes con imágenes:       1471
Pacientes con exoma:          645
Pacientes con ambas fuentes:  509
Pacientes solo imágenes:      962
Pacientes solo exoma:         136
Pacientes sin ninguna fuente: 7012
Archivo generado: /content/drive/MyDrive/Investigación_Parkinson/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados/paciente_master_index.csv


## Note for Notebook 5

From here, the dashboard (Notebook 5) can replace loading `participant_index.csv` + `imagenes_index.csv` + `exomas_index.csv` with this single read:

```python
paciente_master_index = pd.read_csv(RUTA_SALIDA_MASTER_INDEX)
fila = paciente_master_index[paciente_master_index["PATNO"] == patno]
```

Everything that previously required cross-referencing three files (clinical info, image summary, exome location) is now in a single row. The actual extraction of DICOM/VCF remains delegated to the visualization notebook (it uses `Ruta_interna_rar` / `Archivo_TAR` + `Archivo_VCF` as a starting point), which was not touched or modified here.